# Final Ensemble — Custom Architecture Models
## Computer Vision Project — UC3M — CodaBench Submission

This notebook produces the **final CodaBench prediction file (`output_custom.csv`)** for the custom architecture ensemble.

### Overview
This ensemble combines the three custom models trained from scratch:
1. **SmallVGG** (`custom1`) — VGG-inspired network with BatchNorm and Global Average Pooling, trained on Green Channel CLAHE images. Best single custom model: **Val AUC 0.7629**
2. **CustomLeNet** (`custom2`) — LeNet-style network with AvgPool, trained on Green Channel images. **Val AUC 0.6231** (weakest, but adds diversity)
3. **SmallResNet** (`custom3`) — Residual network with skip connections, trained on Green Channel CLAHE images. **Val AUC 0.7287**

### Why Ensemble Custom Models?
Even though individual custom models are weaker than fine-tuned pretrained models, an ensemble can still improve over any single model by:
- Averaging out individual model errors
- Combining different architectural inductive biases (VGG-style, LeNet-style, ResNet-style)
- Leveraging that each model uses slightly different preprocessing (different CLAHE parameters, etc.)

### Strategy: Stacking with Logistic Regression Meta-Learner
Same stacking strategy as the FT ensemble:
1. Split the 500-image validation set 50/50 → meta-train / meta-val
2. Collect base model predictions on meta-train → feature matrix
3. Train Logistic Regression meta-learner
4. Evaluate on meta-val → ensemble AUC
5. Apply to test set → `output_custom.csv`

## 1. Imports and Configuration

Standard setup identical to all other notebooks in the project. Fixed seed for reproducibility.

In [ ]:
import os
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm
from torchvision import transforms

from skimage import io, transform, color
import cv2

from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression

import random

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Dataset — `RetinopathyDataset`

The same dataset class used during individual custom model training. Right-eye mirroring and binary label conversion are applied consistently.

In [ ]:
class RetinopathyDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0):
        self.dataset = pd.read_csv(csv_file, dtype={'id': str, 'eye': int, 'label': int})

        if maxSize > 0:
            idx = np.random.RandomState(seed=42).permutation(len(self.dataset))
            self.dataset = self.dataset.iloc[idx[:maxSize]].reset_index(drop=True)

        self.root_dir = root_dir
        self.img_dir = os.path.join(root_dir, 'images')
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        row = self.dataset.iloc[idx]
        img_path = os.path.join(self.img_dir, row.id + '.jpg')
        image = io.imread(img_path)

        if row.eye == 1:
            image = image[:, ::-1, :]

        label = int(row.label > 0)
        sample = {'image': image, 'label': label}

        if self.transform:
            sample = self.transform(sample)
        return sample

## 3. Preprocessing Transforms

The custom models use a **domain-specific green channel preprocessing pipeline** instead of standard RGB + ImageNet normalization:

- **`GreenChannelCLAHE`:** extracts the green channel (highest contrast for retinal microstructure in DR) and applies CLAHE to enhance local contrast. Used by SmallVGG and SmallResNet.
- **`GreenChannel`:** extracts the green channel without CLAHE enhancement. Used by the LeNet model.

**Critical:** the preprocessing must be identical to what each model was trained with. Mismatched preprocessing would invalidate the base model predictions.

In [ ]:
class CropByEye(object):
    def __init__(self, threshold, border):
        self.threshold = threshold
        self.border = (border, border) if isinstance(border, int) else border

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        gray = color.rgb2gray(image)
        _, mask = cv2.threshold(gray, self.threshold, 1, cv2.THRESH_BINARY)
        sidx = np.nonzero(mask)
        if len(sidx[0]) < 20:
            return sample
        minx = max(sidx[1].min() - self.border[1], 0)
        maxx = min(sidx[1].max() + self.border[1], w)
        miny = max(sidx[0].min() - self.border[0], 0)
        maxy = min(sidx[0].max() + self.border[0], h)
        image = image[miny:maxy, minx:maxx]
        return {'image': image, 'label': label}


class Rescale(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        if h > w:
            new_h = int(self.size * h / w)
            new_w = self.size
        else:
            new_h = self.size
            new_w = int(self.size * w / h)
        image = transform.resize(image, (new_h, new_w))
        return {'image': image, 'label': label}


class CenterCrop(object):
    def __init__(self, size):
        self.size = size

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        top  = max((h - self.size) // 2, 0)
        left = max((w - self.size) // 2, 0)
        image = image[top:top+self.size, left:left+self.size]
        return {'image': image, 'label': label}


class ToTensor(object):
    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        image = image.transpose((2, 0, 1))
        return {'image': torch.tensor(image, dtype=torch.float32),
                'label': torch.tensor(label, dtype=torch.float32)}


class Normalize(object):
    def __init__(self, mean, std):
        self.mean = np.array(mean)
        self.std  = np.array(std)

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        mean = torch.tensor(self.mean, dtype=image.dtype)
        std  = torch.tensor(self.std,  dtype=image.dtype)
        image = (image - mean[:, None, None]) / std[:, None, None]
        return {'image': image, 'label': label}


class GreenChannelCLAHE(object):
    def __init__(self, clip_limit=1.5, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        if image.dtype != np.uint8:
            image = (image * 255).astype(np.uint8) if image.max() <= 1.0 else image.astype(np.uint8)
        green = self.clahe.apply(image[:, :, 1])
        image = np.stack([green, green, green], axis=-1).astype(np.float32) / 255.0
        return {'image': image, 'label': label}


class GreenChannel(object):
    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        if image.dtype != np.float32:
            image = image.astype(np.float32)
        if image.max() > 1.0:
            image = image / 255.0
        green = image[:, :, 1]
        image = np.stack([green, green, green], axis=-1)
        return {'image': image, 'label': label}

## 4. Transform Pipelines — Per Model

Each custom model was trained with a slightly different preprocessing pipeline. We must use the **exact same pipeline** during inference to ensure the model receives expected input distributions:

- **SmallVGG (custom1):** CropByEye → GreenChannelCLAHE(clip=1.0, tile=12×12) → Rescale(320) → CenterCrop(256) → ToTensor → Normalize(green_stats)
- **LeNet (custom2):** CropByEye → Rescale(256) → CenterCrop(224) → GreenChannel → ToTensor → Normalize(ImageNet_stats)
- **SmallResNet (custom3):** CropByEye → GreenChannelCLAHE(clip=1.5, tile=8×8) → Rescale(300) → CenterCrop(256) → ToTensor → Normalize(green_stats)

In [ ]:
# Green channel normalization statistics
green_mean = [0.1703, 0.1703, 0.1703]
green_std  = [0.1742, 0.1742, 0.1742]

# ImageNet statistics (used by LeNet which doesn't apply CLAHE)
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

# SmallVGG (custom1) transform: CLAHE clip=1.0, tile=12×12, size=256
val_transform_vgg = transforms.Compose([
    CropByEye(0.08, 1),
    GreenChannelCLAHE(clip_limit=1.0, tile_grid_size=(12, 12)),
    Rescale(320),
    CenterCrop(256),
    ToTensor(),
    Normalize(green_mean, green_std)
])

# CustomLeNet (custom2) transform: plain green channel, 224×224
val_transform_lenet = transforms.Compose([
    CropByEye(0.10, 1),
    Rescale(256),
    CenterCrop(224),
    GreenChannel(),
    ToTensor(),
    Normalize(imagenet_mean, imagenet_std)
])

# SmallResNet (custom3) transform: CLAHE clip=1.5, tile=8×8, size=256
val_transform_resnet = transforms.Compose([
    CropByEye(0.10, 1),
    GreenChannelCLAHE(clip_limit=1.5, tile_grid_size=(8, 8)),
    Rescale(300),
    CenterCrop(256),
    ToTensor(),
    Normalize(green_mean, green_std)
])

## 5. Load Datasets

Three separate dataset instances are created for validation and test sets — one per base model, each using the model's specific transform pipeline. The training set is not needed here (we only run inference).

In [ ]:
data_dir = "/content/drive/MyDrive/CV_b/Project_2/data"

# --- SmallVGG datasets ---
val_dataset_vgg = RetinopathyDataset(
    os.path.join(data_dir, "val.csv"), data_dir, transform=val_transform_vgg)
test_dataset_vgg = RetinopathyDataset(
    os.path.join(data_dir, "test.csv"), data_dir, transform=val_transform_vgg)

# --- LeNet datasets ---
val_dataset_lenet = RetinopathyDataset(
    os.path.join(data_dir, "val.csv"), data_dir, transform=val_transform_lenet)
test_dataset_lenet = RetinopathyDataset(
    os.path.join(data_dir, "test.csv"), data_dir, transform=val_transform_lenet)

# --- SmallResNet datasets ---
val_dataset_resnet = RetinopathyDataset(
    os.path.join(data_dir, "val.csv"), data_dir, transform=val_transform_resnet)
test_dataset_resnet = RetinopathyDataset(
    os.path.join(data_dir, "test.csv"), data_dir, transform=val_transform_resnet)

## 6. Validation Set Split — Meta-Train / Meta-Val

The 500-image validation set is split 50/50 into meta-train (250 images) and meta-val (250 images). A fixed random seed ensures the split is identical across all three model datasets — this is essential so that `X_meta_train` rows correspond to the same images across all models.

**Why use validation data for meta-training?**  
Each custom model was trained only on the 2000-image training set — they have never seen the validation set. Using validation predictions to train the meta-learner is therefore valid: there is no data leakage.

In [ ]:
val_size        = len(val_dataset_vgg)
meta_train_size = val_size // 2
meta_val_size   = val_size - meta_train_size

# Same split indices for all three model datasets (fixed seed)
generator = torch.Generator().manual_seed(seed)
meta_train_vgg,    meta_val_vgg    = random_split(val_dataset_vgg,    [meta_train_size, meta_val_size], generator=generator)
generator = torch.Generator().manual_seed(seed)
meta_train_lenet,  meta_val_lenet  = random_split(val_dataset_lenet,  [meta_train_size, meta_val_size], generator=generator)
generator = torch.Generator().manual_seed(seed)
meta_train_resnet, meta_val_resnet = random_split(val_dataset_resnet, [meta_train_size, meta_val_size], generator=generator)

# Data loaders
mt_loader_vgg    = DataLoader(meta_train_vgg,    batch_size=64, shuffle=False)
mv_loader_vgg    = DataLoader(meta_val_vgg,      batch_size=64, shuffle=False)
mt_loader_lenet  = DataLoader(meta_train_lenet,  batch_size=64, shuffle=False)
mv_loader_lenet  = DataLoader(meta_val_lenet,    batch_size=64, shuffle=False)
mt_loader_resnet = DataLoader(meta_train_resnet, batch_size=64, shuffle=False)
mv_loader_resnet = DataLoader(meta_val_resnet,   batch_size=64, shuffle=False)

test_loader_vgg    = DataLoader(test_dataset_vgg,    batch_size=64, shuffle=False)
test_loader_lenet  = DataLoader(test_dataset_lenet,  batch_size=64, shuffle=False)
test_loader_resnet = DataLoader(test_dataset_resnet, batch_size=64, shuffle=False)

print(f"Meta-train: {meta_train_size} | Meta-val: {meta_val_size}")

## 7. Custom Model Architectures

The three custom model architectures are defined here exactly as they were during training. Weights will be loaded from saved checkpoints.

In [ ]:
import torch.nn.functional as F

# ─── SmallVGG ─────────────────────────────────────────────────────────────────
class SmallVGG(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.Conv2d(96, 96, 3, padding=1), nn.BatchNorm2d(96), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(96, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.classifier(self.gap(self.features(x)))


# ─── CustomNetLeNet ────────────────────────────────────────────────────────────
class CustomNetLeNet(nn.Module):
    def __init__(self, img_size=224):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)
        self.pool  = nn.AvgPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        with torch.no_grad():
            x = torch.zeros(1, 3, img_size, img_size)
            x = self.pool(F.relu(self.conv1(x)))
            x = self.pool(F.relu(self.conv2(x)))
            flat_dim = x.flatten(1).shape[1]
        self.fc1 = nn.Linear(flat_dim, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 1)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.flatten(1)
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))


# ─── SmallResNet ───────────────────────────────────────────────────────────────
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.shortcut(x), inplace=True)

class SmallResNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem   = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True))
        self.layer1 = ResidualBlock(32,  32,  stride=1)
        self.layer2 = ResidualBlock(32,  64,  stride=2)
        self.layer3 = ResidualBlock(64,  96,  stride=2)
        self.layer4 = ResidualBlock(96,  128, stride=2)
        self.gap    = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(64, 1))

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.classifier(self.gap(x))

## 8. Load Saved Checkpoints

Pre-trained weights from the best-epoch checkpoints of each custom model are loaded. The models are moved to GPU and set to evaluation mode.

In [ ]:
models_dir = "/content/drive/MyDrive/CV_b/Project_2/Models"

# SmallVGG
vgg_model = SmallVGG().to(device)
vgg_model.load_state_dict(torch.load(os.path.join(models_dir, "best_model_custom_git.pth")))
vgg_model.eval()

# CustomLeNet
lenet_model = CustomNetLeNet(img_size=224).to(device)
lenet_model.load_state_dict(torch.load(os.path.join(models_dir, "best_customnet2.pth")))
lenet_model.eval()

# SmallResNet
resnet_model = SmallResNet().to(device)
resnet_model.load_state_dict(torch.load(os.path.join(models_dir, "best_customnet3.pth")))
resnet_model.eval()

print("All custom models loaded successfully.")

## 9. Inference Function

The `predict` function runs a model in eval mode and returns sigmoid probabilities as an (N, 1) column vector. Test-Time Augmentation (TTA) with horizontal flips is applied for SmallVGG and SmallResNet (the models that used TTA during training validation).

In [ ]:
def predict(model, loader):
    """Standard inference without TTA."""
    model.eval()
    outputs = []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['image'].to(device)
            preds  = torch.sigmoid(model(inputs).squeeze())
            outputs.extend(preds.cpu().numpy())
    return np.array(outputs).reshape(-1, 1)


def predict_tta(model, loader):
    """Inference with horizontal-flip TTA (average of original + flipped)."""
    model.eval()
    outputs = []
    with torch.no_grad():
        for batch in loader:
            inputs = batch['image'].to(device)
            p1 = torch.sigmoid(model(inputs).view(-1))
            p2 = torch.sigmoid(model(torch.flip(inputs, dims=[3])).view(-1))
            probs = ((p1 + p2) / 2.0)
            outputs.extend(probs.cpu().numpy())
    return np.array(outputs).reshape(-1, 1)

## 10. Build Meta-Feature Matrices

Base model predictions are collected on meta-train and meta-val splits. SmallVGG and SmallResNet use TTA; LeNet uses standard inference (TTA had negligible impact on the simpler architecture).

The feature matrices have shape (250, 3) — one row per image, one column per base model.

In [ ]:
# ─── META-TRAIN FEATURES ───────────────────────────────────────────────────────
vgg_mt    = predict_tta(vgg_model,    mt_loader_vgg)
lenet_mt  = predict(lenet_model,      mt_loader_lenet)
resnet_mt = predict_tta(resnet_model, mt_loader_resnet)

X_meta_train = np.concatenate([vgg_mt, lenet_mt, resnet_mt], axis=1)
y_meta_train = np.array([meta_train_vgg[i]['label'] for i in range(len(meta_train_vgg))])

print(f"X_meta_train shape: {X_meta_train.shape}")  # (250, 3)
print(f"y_meta_train shape: {y_meta_train.shape}")

# ─── META-VAL FEATURES ─────────────────────────────────────────────────────────
vgg_mv    = predict_tta(vgg_model,    mv_loader_vgg)
lenet_mv  = predict(lenet_model,      mv_loader_lenet)
resnet_mv = predict_tta(resnet_model, mv_loader_resnet)

X_meta_val = np.concatenate([vgg_mv, lenet_mv, resnet_mv], axis=1)
y_meta_val = np.array([meta_val_vgg[i]['label'] for i in range(len(meta_val_vgg))])

print(f"X_meta_val shape:   {X_meta_val.shape}")

## 11. Train Logistic Regression Meta-Learner

A Logistic Regression is trained on the 250×3 meta-train feature matrix. The learned weights indicate the relative reliability of each custom model's predictions:
- **High weight** → the meta-learner trusts that model's probability more
- **Low or negative weight** → the model's predictions may be noisy or anticorrelated

After training, the ensemble AUC on meta-val is reported.

In [ ]:
meta_model = LogisticRegression(max_iter=1000, random_state=seed)
meta_model.fit(X_meta_train, y_meta_train)

val_preds = meta_model.predict_proba(X_meta_val)[:, 1]
ensemble_auc = roc_auc_score(y_meta_val, val_preds)

print(f"Meta-learner Val AUC: {ensemble_auc:.4f}")
print(f"Model weights: {dict(zip(['SmallVGG', 'LeNet', 'SmallResNet'], meta_model.coef_[0]))}")

## 12. Final Test Set Predictions

The trained meta-learner is applied to the 1000-image test set. Base model predictions are collected, concatenated, and fed to the Logistic Regression to produce the final ensemble probability score per image.

In [ ]:
# Base model predictions on test set
vgg_test    = predict_tta(vgg_model,    test_loader_vgg)
lenet_test  = predict(lenet_model,      test_loader_lenet)
resnet_test = predict_tta(resnet_model, test_loader_resnet)

X_test_meta = np.concatenate([vgg_test, lenet_test, resnet_test], axis=1)
final_preds = meta_model.predict_proba(X_test_meta)[:, 1]

print(f"Test predictions shape: {final_preds.shape}")
print(f"Prediction range: [{final_preds.min():.4f}, {final_preds.max():.4f}]")

## 13. Save Predictions to CSV

The final ensemble predictions are saved to `custom_ensemble_submission.csv`. This file will be renamed to `output_custom.csv` and packaged into the CodaBench submission ZIP by `generate_submission.ipynb`.

In [ ]:
import pandas as pd

submission = pd.DataFrame({"prediction": final_preds})
submission.to_csv("custom_ensemble_submission.csv", index=False)

# Also save to Google Drive for persistence
submission.to_csv("/content/drive/MyDrive/custom_ensemble_submission.csv", index=False)

print(f"Saved {len(final_preds)} predictions.")
print("Shape check:", final_preds.reshape(-1, 1).shape)
print("Range check:", final_preds.min() >= 0 and final_preds.max() <= 1)